# Domain shift / regime shift analysis

Questa analisi non usa il modello ST-GNN e non richiede training. Misura se la produzione reale degli impianti cambia nel tempo rispetto a una baseline meteo/PVGIS, mantenendo separate due popolazioni con diverso livello di affidabilita' metrologica.

**Impianti con kWp reale.** Per i 94 impianti con potenza nominale da registro si usa `PR_PVGIS = actual_kwh / (kWp * PVGIS_POA)`. Questa e' la misura principale per trend, Kendall tau, `decline_class` e level shift.

**Impianti senza kWp reale.** Per gli impianti senza potenza nominale reale non si calcola un PR assoluto. Si usa invece un `relative_index` normalizzato intra-impianto, costruito da `apparent_capacity = actual_kwh / expected_pvgis_per_kwp` e poi diviso per una baseline dello stesso impianto. Questo indice serve solo a rilevare cambiamenti temporali dentro lo stesso impianto, non a confrontare impianti diversi.

I downshift sono interpretati come *apparent performance loss* o *regime shift*, non automaticamente come degradazione fisica. Possibili cause: soiling, guasti, availability losses, curtailment, clipping, problemi dati, cambiamenti operativi, stagionalita' residua o degradazione fisica.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs' / 'domain_shift_trend'
OUT

## Run dello script

Esegui questa cella sul server dove sono presenti Sentinel/PVGIS e i CSV di mapping. Lo script rigenera sia l'analisi real-kWp (`PR_PVGIS`) sia l'analisi esplorativa non-real-kWp (`relative_index`) in output separati. Se gli output sono gia' stati prodotti, puoi saltare la cella e caricare direttamente le sezioni successive.

In [ ]:
cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'analyze_domain_shift_trend.py'),
    '--out-dir', str(OUT),
    '--kwp-mode', 'real-only',
    '--analyze-non-real-kwp',
    '--plausible-pr-min', '0.2',
    '--plausible-pr-max', '2.0',
    '--overprediction-bias-pct', '5.0',
]
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT) + os.pathsep + env.get('PYTHONPATH', '')

print(' '.join(cmd))
res = subprocess.run(cmd, cwd=ROOT, env=env, text=True, capture_output=True)
stdout = res.stdout or ''
stderr = res.stderr or ''
if stdout:
    print(stdout)
if stderr:
    print(stderr, file=sys.stderr)
if res.returncode != 0:
    diagnostic_tail = '\n'.join((stderr or stdout).splitlines()[-80:])
    raise RuntimeError(
        f'Script failed with exit code {res.returncode}\n'
        f'--- diagnostic tail ---\n{diagnostic_tail}'
    )

## Summary numerico

Questa sezione carica i riepiloghi descrittivi. `summary.json` riguarda la baseline real-kWp; `combined_real_and_relative_summary.json`, se presente, affianca real-kWp e non-real-kWp solo come riepilogo descrittivo, senza confrontare direttamente i valori delle metriche.

In [ ]:
with open(OUT / 'summary.json', encoding='utf-8') as f:
    summary = json.load(f)

combined_path = OUT / 'combined_real_and_relative_summary.json'
combined_summary = None
if combined_path.exists():
    with open(combined_path, encoding='utf-8') as f:
        combined_summary = json.load(f)

print('summary.json')
display(summary)
if combined_summary is not None:
    print('\ncombined_real_and_relative_summary.json')
    display(combined_summary)

In [ ]:
fleet_trends = pd.read_csv(OUT / 'fleet_trend_summary.csv')
plant_trends = pd.read_csv(OUT / 'plant_trend_summary.csv')
fleet_monthly = pd.read_csv(OUT / 'fleet_monthly_performance.csv', parse_dates=['date'])
plant_monthly = pd.read_csv(OUT / 'plant_monthly_performance.csv', parse_dates=['date'])
real_kwp_trends = pd.read_csv(OUT / 'real_kwp_plant_pr_trends.csv')

print(f'real_kwp_plant_pr_trends.csv shape: {real_kwp_trends.shape}')
fleet_trends

## Grafici real-kWp generati dallo script

In [ ]:
for name in [
    'fleet_pvgis_pr_trend.png',
    'top_decreasing_plants_pvgis_pr.png',
    'individual_candidate_plant_trends.png',
]:
    path = OUT / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Missing: {path}')

## Impianti real-kWp con trend PR_PVGIS decrescente piu' forte

In [ ]:
cols = [
    'metric', 'plant', 'plant_id', 'n_points', 'mean_value',
    'slope_per_year', 'relative_change_pct_per_year', 'p_value',
    'kendall_tau', 'decreasing'
]

pr_trends = plant_trends[plant_trends['metric'] == 'pr_pvgis_monthly'].copy()
pr_trends.sort_values('slope_per_year')[cols].head(20)

## Andamento fleet mensile real-kWp

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fleet_monthly['date'], fleet_monthly['weighted_pr_pvgis'], 'o-', label='weighted PR PVGIS')
ax.plot(fleet_monthly['date'], fleet_monthly['median_pr_pvgis'], 's--', label='median plant PR')
ax.axhline(fleet_monthly['weighted_pr_pvgis'].mean(), color='0.3', linewidth=1, alpha=0.6)
ax.set_xlabel('Mese')
ax.set_ylabel('actual / (kWp * PVGIS POA)')
ax.set_title('Performance normalizzata PVGIS - flotta')
ax.legend()
plt.tight_layout()

## Distribuzione completa dei trend real-kWp (PR_PVGIS)

Non solo i candidati forti: classifica ogni impianto real-kWp in bin di `relative_change_pct_per_year` e in classi di monotonicita' basate su Kendall tau. I cali deboli sono *apparent weak / mild performance loss*, non degradazione fisica confermata: 1 anno incompleto e pochi punti mensili per impianto non separano degrado debole da stagionalita' residua, soiling, availability, curtailment, clipping o problemi nei dati.

In [ ]:
# Distribuzione completa dei trend per impianto
# Legge i file generati dallo script:
#   - all_plant_pr_trends.csv
#   - all_plant_decline_classes.csv
#   - decline_distribution_summary.json
#   - all_plant_relative_change_histogram.png
import json
from IPython.display import Image, display

dist_path = OUT / "decline_distribution_summary.json"
all_path = OUT / "all_plant_pr_trends.csv"
classes_path = OUT / "all_plant_decline_classes.csv"
hist_path = OUT / "all_plant_relative_change_histogram.png"

if not dist_path.exists():
    print(
        "decline_distribution_summary.json mancante. "
        "Rilancia la cella di run dello script per generarlo."
    )
else:
    with open(dist_path, encoding="utf-8") as f:
        dist = json.load(f)

    print(f"n_plants_analyzed: {dist['n_plants_analyzed']}")
    print(f"alpha: {dist['alpha']}")
    print(
        f"strong tau threshold: {dist['monotonic_strong_tau_threshold']}, "
        f"directional tau threshold: {dist['monotonic_directional_tau_threshold']}"
    )

    counts = pd.DataFrame(
        {
            "decline_class": list(dist["decline_class_counts"].keys()),
            "count": list(dist["decline_class_counts"].values()),
            "pct": [dist["decline_class_pct"][k] for k in dist["decline_class_counts"]],
        }
    )
    print("\nDistribuzione classi di decline (relative_change_pct_per_year):")
    display(counts)

    mono = pd.DataFrame(
        {
            "monotonic_class": list(dist["monotonic_class_counts"].keys()),
            "count": list(dist["monotonic_class_counts"].values()),
            "pct": [
                dist["monotonic_class_pct"][k] for k in dist["monotonic_class_counts"]
            ],
        }
    )
    print("\nDistribuzione classi di monotonicita' (Kendall tau):")
    display(mono)

    print(
        "\nn_significant (p < alpha AND slope/year < 0): "
        f"{dist['n_significant_p_lt_alpha_and_negative_slope']} "
        f"({dist['pct_significant_p_lt_alpha_and_negative_slope']:.1f}%)"
    )

    q = dist.get("relative_change_pct_per_year_quantiles", {})
    if q:
        qdf = pd.DataFrame({"quantile": list(q.keys()), "rel_pct_per_year": list(q.values())})
        print("\nQuantili di relative_change_pct_per_year (%/anno):")
        display(qdf)

    print("\nCAVEAT:")
    print(dist["interpretation_caveat"])

    if all_path.exists():
        all_plants = pd.read_csv(all_path)
        print(f"\nall_plant_pr_trends.csv shape: {all_plants.shape}")
        print("Top 20 piu' negativi (relative_change_pct_per_year):")
        display(all_plants.head(20))
        print("\nBottom 10 (piu' positivi o stabili):")
        display(all_plants.tail(10))

    if classes_path.exists():
        cls = pd.read_csv(classes_path)
        weak_mild = cls[
            cls["decline_class"].isin(["weak_decline_1_2", "mild_decline_2_3"])
        ]
        print(
            f"\nImpianti in weak_decline_1_2 + mild_decline_2_3: {len(weak_mild)}"
        )
        if not weak_mild.empty:
            display(
                weak_mild[
                    [
                        "plant",
                        "plant_id",
                        "upn",
                        "valid_months",
                        "pr_mean",
                        "relative_change_pct_per_year",
                        "p_value",
                        "kendall_tau",
                        "decline_class",
                        "monotonic_class",
                    ]
                ].sort_values("relative_change_pct_per_year")
            )

    if hist_path.exists():
        display(Image(filename=str(hist_path)))


## Level shift / regime shift real-kWp

Questa sezione cerca cambi di livello prestazionale nei 94 impianti con kWp reale usando `PR_PVGIS`: prima vs seconda meta' e migliore punto di rottura. E' utile per individuare casi in cui un modello trainato sulla prima parte dell'anno potrebbe sovrastimare la produzione nella seconda. Anche qui il downshift e' *apparent performance loss*, non diagnosi automatica di degradazione fisica.

In [ ]:
# Analisi di level shift / regime shift per impianti real-kWp
# Legge:
#   - real_kwp_plant_pr_trends.csv
#   - plant_level_shift_candidates.csv
#   - level_shift_summary.json
#   - level_shift_*.png
import json
from IPython.display import Image, display

shift_path = OUT / "level_shift_summary.json"
table_path = OUT / "real_kwp_plant_pr_trends.csv"
candidates_path = OUT / "plant_level_shift_candidates.csv"

if not shift_path.exists():
    print(
        "level_shift_summary.json mancante. "
        "Rilancia la cella di run dello script per generarlo."
    )
else:
    with open(shift_path, encoding="utf-8") as f:
        lvl = json.load(f)
    print(f"n_plants_analyzed: {lvl['n_plants_analyzed']}")

    half = pd.DataFrame(
        {
            "half_shift_class": list(lvl["half_shift_class_counts"].keys()),
            "count": list(lvl["half_shift_class_counts"].values()),
            "pct": [lvl["half_shift_class_pct"][k] for k in lvl["half_shift_class_counts"]],
        }
    )
    print("\nHalf-split shift classes:")
    display(half)

    brk = pd.DataFrame(
        {
            "best_break_shift_class": list(lvl["best_break_shift_class_counts"].keys()),
            "count": list(lvl["best_break_shift_class_counts"].values()),
            "pct": [
                lvl["best_break_shift_class_pct"][k]
                for k in lvl["best_break_shift_class_counts"]
            ],
        }
    )
    print("\nBest-break shift classes:")
    display(brk)

    print(
        "\npossible_step_change_with_plateau: "
        f"{lvl['n_possible_step_change_with_plateau']} "
        f"({lvl['pct_possible_step_change_with_plateau']:.1f}%)"
    )
    print(
        "  downshift but NOT monotonic decline: "
        f"{lvl.get('n_downshift_but_not_monotonic_decline', lvl.get('n_plateau_and_not_monotonic_decline'))}"
    )
    print(
        "  downshift but NOT significant slope: "
        f"{lvl.get('n_downshift_but_not_significant_slope', lvl.get('n_plateau_and_not_significant_decline'))}"
    )
    print(
        "  expected overprediction risk if trained pre-break "
        f"(threshold={lvl.get('overprediction_bias_pct_threshold')}%): "
        f"{lvl.get('n_overprediction_risk_if_trained_pre_break')}"
    )
    if lvl["plateau_candidates_by_decline_class"]:
        by_dc = pd.DataFrame(
            {
                "decline_class": list(lvl["plateau_candidates_by_decline_class"].keys()),
                "count": list(lvl["plateau_candidates_by_decline_class"].values()),
            }
        )
        print("\nPlateau candidates by existing decline_class:")
        display(by_dc)

    print("\nCAVEAT:")
    print(lvl["interpretation_caveat"])

    if table_path.exists():
        tbl = pd.read_csv(table_path)
        cols = [
            c
            for c in (
                "plant",
                "plant_id",
                "upn",
                "valid_months",
                "pr_mean",
                "pr_first",
                "pr_last",
                "relative_change_pct_per_year",
                "kendall_tau",
                "decline_class",
                "monotonic_class",
                "first_half_pr_mean",
                "second_half_pr_mean",
                "half_delta_pct",
                "half_shift_class",
                "best_break_month",
                "pre_break_mean",
                "post_break_mean",
                "break_delta_pct",
                "post_break_cv",
                "best_break_shift_class",
                "possible_step_change_with_plateau",
                "expected_bias_pct_if_train_pre_break",
            )
            if c in tbl.columns
        ]
        print(f"\nreal_kwp_plant_pr_trends.csv shape: {tbl.shape}")
        print("Top 20 by most negative break_delta_pct:")
        display(tbl.sort_values("break_delta_pct").head(20)[cols])

    if candidates_path.exists():
        cand = pd.read_csv(candidates_path)
        print(f"\nplant_level_shift_candidates.csv shape: {cand.shape}")
        display(cand.head(20))

    for name in (
        "level_shift_half_delta_pct_histogram.png",
        "level_shift_break_delta_pct_histogram.png",
        "level_shift_scatter_slope_vs_half.png",
        "level_shift_scatter_tau_vs_break.png",
    ):
        p = OUT / name
        if p.exists():
            display(Image(filename=str(p)))

## Impianti senza kWp reale: metodologia `relative_index`

Per gli impianti senza potenza nominale reale non e' possibile calcolare un `PR_PVGIS` assoluto affidabile. La pipeline usa quindi un indice relativo normalizzato per singolo impianto:

`apparent_capacity_i,m = actual_kwh_i,m / expected_pvgis_per_kwp_i,m`

`relative_index_i,m = apparent_capacity_i,m / baseline_apparent_capacity_i`

La baseline predefinita e' la mediana dei primi 3 mesi validi. Questa scelta e' robusta a un singolo mese anomalo e ancora l'indice al comportamento iniziale; se il numero di mesi validi non consente questa baseline ma l'impianto supera comunque la soglia minima scelta nello script, si usa la mediana dei mesi validi e il caso viene marcato con `baseline_method` e `baseline_n_months`.

`relative_index` e' interpretabile solo intra-impianto: consente di dire che un impianto peggiora rispetto al proprio comportamento storico, ma non che un impianto performa meglio o peggio di un altro. Le analisi di trend, Kendall tau, half-shift, best-break, plateau e bias di forecasting sono quindi esplorative e non vanno lette come degradazione fisica automatica.

In [ ]:
# Analisi esplorativa per impianti senza kWp reale (relative_index intra-impianto)
# Legge:
#   - non_real_kwp_relative_trends.csv
#   - non_real_kwp_level_shift_candidates.csv
#   - non_real_kwp_relative_summary.json
#   - combined_real_and_relative_summary.json
#   - histogram_relative_change_pct_per_year_non_real.png
#   - histogram_best_break_delta_pct_non_real.png
#   - scatter_kendall_tau_vs_best_break_delta_pct_non_real.png
#   - scatter_slope_vs_half_delta_pct_non_real.png
#   - top20_level_shift_candidates_non_real.png
import json
from IPython.display import Image, display

rel_summary_path = OUT / "non_real_kwp_relative_summary.json"
rel_trends_path = OUT / "non_real_kwp_relative_trends.csv"
rel_cand_path = OUT / "non_real_kwp_level_shift_candidates.csv"
combined_path = OUT / "combined_real_and_relative_summary.json"

if not rel_summary_path.exists():
    print(
        "non_real_kwp_relative_summary.json mancante. "
        "Rilancia la cella di run dello script sul server dati."
    )
else:
    with open(rel_summary_path, encoding="utf-8") as f:
        rel = json.load(f)

    print(f"n_non_real_kwp_plants_total:    {rel['n_non_real_kwp_plants_total']}")
    print(f"n_non_real_kwp_plants_analyzed: {rel['n_non_real_kwp_plants_analyzed']}")
    print(f"n_excluded_insufficient_data:   {rel['n_excluded_insufficient_data']}")
    print(
        f"excluded detail - no_data: {rel['n_excluded_no_data']}, "
        f"too_few_months: {rel['n_excluded_too_few_months']}, "
        f"no_baseline: {rel['n_excluded_no_baseline']}"
    )
    print(
        f"baseline strategy: {rel['baseline_strategy_requested']} "
        f"(n_months={rel['baseline_n_months_requested']})"
    )

    df_decl = pd.DataFrame(
        {
            "decline_class": list(rel["decline_class_counts"].keys()),
            "count": list(rel["decline_class_counts"].values()),
            "pct": [rel["decline_class_pct"][k] for k in rel["decline_class_counts"]],
        }
    )
    print("\nTrend relativi non-real-kWp: decline classes")
    display(df_decl)

    df_mono = pd.DataFrame(
        {
            "monotonic_class": list(rel["monotonic_class_counts"].keys()),
            "count": list(rel["monotonic_class_counts"].values()),
            "pct": [rel["monotonic_class_pct"][k] for k in rel["monotonic_class_counts"]],
        }
    )
    print("\nTrend relativi non-real-kWp: Kendall tau classes")
    display(df_mono)

    df_half = pd.DataFrame(
        {
            "half_shift_class": list(rel["half_shift_class_counts"].keys()),
            "count": list(rel["half_shift_class_counts"].values()),
            "pct": [rel["half_shift_class_pct"][k] for k in rel["half_shift_class_counts"]],
        }
    )
    print("\nLevel shift non-real-kWp: first-half vs second-half")
    display(df_half)

    df_brk = pd.DataFrame(
        {
            "best_break_shift_class": list(rel["best_break_shift_class_counts"].keys()),
            "count": list(rel["best_break_shift_class_counts"].values()),
            "pct": [rel["best_break_shift_class_pct"][k] for k in rel["best_break_shift_class_counts"]],
        }
    )
    print("\nLevel shift non-real-kWp: best break")
    display(df_brk)

    print(
        f"\npossible_step_change_with_plateau: "
        f"{rel['n_possible_step_change_with_plateau']}"
    )
    print(
        f"downshift but NOT monotonic decline: "
        f"{rel['n_downshift_but_not_monotonic_decline']}"
    )
    print(
        f"downshift but NOT significant slope: "
        f"{rel['n_downshift_but_not_significant_slope']}"
    )
    print(
        f"expected overprediction risk if trained pre-break "
        f"(threshold={rel['overprediction_bias_pct_threshold']}%): "
        f"{rel['n_overprediction_risk_if_trained_pre_break']}"
    )

    print("\nMETHODOLOGY:")
    print(rel["methodology_caveat"])
    print("\nFORECASTING NOTE:")
    print(rel["forecasting_note"])

    if rel_trends_path.exists():
        rt = pd.read_csv(rel_trends_path)
        cols = [
            c
            for c in (
                "plant",
                "plant_id",
                "upn",
                "valid_months",
                "baseline_method",
                "baseline_n_months",
                "relative_index_mean",
                "relative_index_first",
                "relative_index_last",
                "first_to_last_pct",
                "slope_per_month",
                "slope_per_year",
                "relative_change_pct_per_year",
                "p_value",
                "kendall_tau",
                "kendall_p_value",
                "decreasing",
                "decline_class",
                "monotonic_class",
                "first_half_index_mean",
                "second_half_index_mean",
                "half_delta_pct",
                "half_shift_class",
                "best_break_month",
                "pre_break_index_mean",
                "post_break_index_mean",
                "break_delta_abs",
                "break_delta_pct",
                "pre_break_std",
                "post_break_std",
                "post_break_cv",
                "best_break_shift_class",
                "possible_step_change_with_plateau",
                "expected_bias_if_train_pre_break",
                "expected_bias_pct_if_train_pre_break",
            )
            if c in rt.columns
        ]
        print(f"\nnon_real_kwp_relative_trends.csv shape: {rt.shape}")
        print("Top 20 level shift candidates (most negative break_delta_pct):")
        display(rt.sort_values("break_delta_pct").head(20)[cols])

    if rel_cand_path.exists():
        cand = pd.read_csv(rel_cand_path)
        print(f"\nnon_real_kwp_level_shift_candidates.csv shape: {cand.shape}")
        display(cand.head(20))

    if combined_path.exists():
        with open(combined_path, encoding="utf-8") as f:
            comb = json.load(f)
        print("\nCombined descriptive summary (non comparare direttamente i valori):")
        rk = comb["real_kwp"]
        nrk = comb["non_real_kwp_relative_index"]
        print(
            f"  real-kWp PR_PVGIS:       n={rk['n_plants_analyzed']}, "
            f"plateau={rk['level_shift']['n_possible_step_change_with_plateau']}"
        )
        print(
            f"  non-real relative_index: n={nrk['n_plants_analyzed']}, "
            f"plateau={nrk['level_shift']['n_possible_step_change_with_plateau']}, "
            f"overpred_risk={nrk['n_overprediction_risk_if_trained_pre_break']}"
        )
        print("  note:", comb["note"])

    for name in (
        "histogram_relative_change_pct_per_year_non_real.png",
        "histogram_best_break_delta_pct_non_real.png",
        "scatter_kendall_tau_vs_best_break_delta_pct_non_real.png",
        "scatter_slope_vs_half_delta_pct_non_real.png",
        "top20_level_shift_candidates_non_real.png",
    ):
        p = OUT / name
        if p.exists():
            display(Image(filename=str(p)))
        else:
            print(f"Missing: {p}")

## Lettura per la tesi

**Risultati real-kWp.** Sugli impianti con kWp reale, `PR_PVGIS` e' la metrica principale: consente di stimare una perdita prestazionale apparente normalizzata per irraggiamento e potenza nominale. Le classi di trend, Kendall tau e level shift sui 94 impianti sono quindi la parte piu' solida dell'analisi, pur restando limitata da un solo anno incompleto.

**Risultati non-real-kWp.** Sugli impianti senza kWp reale non e' disponibile un PR assoluto. Il `relative_index` permette solo diagnosi temporali intra-impianto: individua downshift, cambi di regime e possibili plateau rispetto al comportamento iniziale/storico dello stesso impianto. Questi risultati sono esplorativi e non devono essere usati per confronti assoluti tra impianti.

**Collegamento al forecasting.** Anche senza kWp reale, un downshift del `relative_index` puo' indicare un cambio di regime rilevante per il modello: un forecaster addestrato su mesi pre-break puo' sovrastimare la produzione nel regime post-break. La colonna `expected_bias_pct_if_train_pre_break` quantifica questa stima in modo descrittivo.

In entrambi i casi, un downshift non e' automaticamente degradazione fisica. Possibili cause includono soiling, guasti, availability losses, curtailment, clipping, problemi dati, cambiamenti operativi, stagionalita' residua o degradazione fisica.

## Report compatto da mandare per controllo

Esegui questa cella sul server dopo aver rigenerato gli output. Il report separa esplicitamente risultati real-kWp (`PR_PVGIS`) e risultati non-real-kWp (`relative_index`) e include shape, prime righe, candidati level shift e note metodologiche.

In [ ]:
def _compact_report(out_dir: Path, top_n: int = 20):
    def _safe_read_csv(path: Path) -> pd.DataFrame:
        if not path.exists() or path.stat().st_size == 0:
            return pd.DataFrame()
        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()

    def _safe_read_json(path: Path) -> dict:
        if not path.exists():
            return {}
        with open(path, encoding='utf-8') as f:
            return json.load(f)

    summary = _safe_read_json(out_dir / 'summary.json')
    rel_summary = _safe_read_json(out_dir / 'non_real_kwp_relative_summary.json')
    combined = _safe_read_json(out_dir / 'combined_real_and_relative_summary.json')

    fleet_trends = _safe_read_csv(out_dir / 'fleet_trend_summary.csv')
    plant_trends = _safe_read_csv(out_dir / 'plant_trend_summary.csv')
    fleet_monthly = _safe_read_csv(out_dir / 'fleet_monthly_performance.csv')
    plant_monthly = _safe_read_csv(out_dir / 'plant_monthly_performance.csv')
    candidate_summary = _safe_read_csv(out_dir / 'plant_candidate_summary.csv')
    real_shift = _safe_read_csv(out_dir / 'real_kwp_plant_pr_trends.csv')
    real_shift_candidates = _safe_read_csv(out_dir / 'plant_level_shift_candidates.csv')
    rel_trends = _safe_read_csv(out_dir / 'non_real_kwp_relative_trends.csv')
    rel_candidates = _safe_read_csv(out_dir / 'non_real_kwp_level_shift_candidates.csv')
    excluded = _safe_read_csv(out_dir / 'excluded_implausible_pr.csv')
    excluded_diag = _safe_read_csv(out_dir / 'excluded_implausible_pr_diagnostics.csv')

    print('=== REAL-KWP SUMMARY.JSON ===')
    print(json.dumps(summary, indent=2))

    if combined:
        print('\n=== COMBINED DESCRIPTIVE SUMMARY ===')
        print(json.dumps(combined, indent=2))

    print('\n=== FLEET TREND SUMMARY: REAL-KWP PR_PVGIS ===')
    print(fleet_trends.to_string(index=False) if not fleet_trends.empty else 'missing')

    if not fleet_monthly.empty:
        print('\n=== FLEET MONTHLY PERFORMANCE: REAL-KWP PR_PVGIS ===')
        fm_cols = [
            c for c in (
                'date', 'n_plants', 'actual_kwh', 'pvgis_expected_kwh',
                'weighted_pr_pvgis', 'median_pr_pvgis'
            ) if c in fleet_monthly.columns
        ]
        print(fleet_monthly[fm_cols].to_string(index=False))

    if not plant_trends.empty:
        pr = plant_trends[plant_trends['metric'] == 'pr_pvgis_monthly'].copy()
        plant_z = plant_trends[plant_trends['metric'] == 'pr_plant_z_monthly'].copy()
        cols = [
            'plant', 'plant_id', 'n_points', 'mean_value', 'first_value', 'last_value',
            'slope_per_year', 'relative_change_pct_per_year', 'p_value',
            'kendall_tau', 'kendall_p_value', 'decreasing'
        ]
        print(f'\n=== TOP {top_n} DECREASING REAL-KWP PLANTS: PR_PVGIS ===')
        print(pr.sort_values('slope_per_year').head(top_n)[cols].to_string(index=False))
        print(f'\n=== TOP {top_n} DECREASING REAL-KWP PLANTS: PR_PLANT_Z ===')
        if plant_z.empty:
            print('missing pr_plant_z_monthly')
        else:
            print(plant_z.sort_values('slope_per_year').head(top_n)[cols].to_string(index=False))

    print(f'\n=== REAL-KWP LEVEL SHIFT: real_kwp_plant_pr_trends.csv shape={real_shift.shape} ===')
    if real_shift.empty:
        print('missing')
    else:
        real_cols = [
            c for c in (
                'plant', 'plant_id', 'upn', 'valid_months', 'pr_mean', 'pr_first', 'pr_last',
                'relative_change_pct_per_year', 'p_value', 'kendall_tau', 'decline_class',
                'monotonic_class', 'first_half_pr_mean', 'second_half_pr_mean',
                'best_break_month', 'pre_break_mean', 'post_break_mean', 'break_delta_pct',
                'post_break_cv', 'best_break_shift_class', 'possible_step_change_with_plateau',
                'expected_bias_pct_if_train_pre_break'
            ) if c in real_shift.columns
        ]
        print(real_shift.sort_values('break_delta_pct').head(top_n)[real_cols].to_string(index=False))

    print(f'\n=== REAL-KWP LEVEL SHIFT CANDIDATES shape={real_shift_candidates.shape} ===')
    print(real_shift_candidates.head(top_n).to_string(index=False) if not real_shift_candidates.empty else 'none/missing')

    print('\n=== NON-REAL-KWP RELATIVE SUMMARY ===')
    print(json.dumps(rel_summary, indent=2) if rel_summary else 'missing')

    print(f'\n=== NON-REAL-KWP RELATIVE TRENDS shape={rel_trends.shape} ===')
    if rel_trends.empty:
        print('missing')
    else:
        rel_cols = [
            c for c in (
                'plant', 'plant_id', 'upn', 'valid_months', 'baseline_method',
                'baseline_n_months', 'relative_index_mean', 'relative_index_first',
                'relative_index_last', 'first_to_last_pct', 'slope_per_year',
                'relative_change_pct_per_year', 'p_value', 'kendall_tau', 'decline_class',
                'monotonic_class', 'first_half_index_mean', 'second_half_index_mean',
                'best_break_month', 'pre_break_index_mean', 'post_break_index_mean',
                'break_delta_pct', 'post_break_cv', 'best_break_shift_class',
                'possible_step_change_with_plateau', 'expected_bias_pct_if_train_pre_break'
            ) if c in rel_trends.columns
        ]
        print(rel_trends.sort_values('break_delta_pct').head(top_n)[rel_cols].to_string(index=False))

    print(f'\n=== NON-REAL-KWP LEVEL SHIFT CANDIDATES shape={rel_candidates.shape} ===')
    print(rel_candidates.head(top_n).to_string(index=False) if not rel_candidates.empty else 'none/missing')

    print('\n=== DATA COVERAGE / SOURCES: REAL-KWP MONTHLY TABLE ===')
    if plant_monthly.empty:
        print('missing plant_monthly_performance.csv')
    else:
        print('plant_monthly rows:', len(plant_monthly))
        print('plants in monthly table:', plant_monthly['plant'].nunique())
        print('kwp_source counts:')
        print(plant_monthly[['plant', 'kwp_source']].drop_duplicates()['kwp_source'].value_counts().to_string())
        print('valid monthly PR_PVGIS points per plant:')
        print(plant_monthly.groupby('plant')['pr_pvgis'].count().describe().to_string())

    print('\n=== EXCLUDED IMPLAUSIBLE PR PLANTS: REAL-KWP ===')
    if excluded.empty:
        print('none')
    else:
        ex_cols = [
            c for c in (
                'plant', 'plant_id', 'mean_pr_pvgis', 'median_pr_pvgis',
                'min_pr_pvgis', 'max_pr_pvgis', 'n_valid_months', 'kwp_used', 'kwp_source'
            ) if c in excluded.columns
        ]
        print(excluded.sort_values('mean_pr_pvgis', ascending=False)[ex_cols].head(40).to_string(index=False))

    print('\n=== WHY PR IS IMPLAUSIBLE: COMPONENT DIAGNOSTICS ===')
    if excluded_diag.empty:
        print('none')
    else:
        diag_cols = [
            c for c in (
                'plant', 'plant_id', 'upn', 'Codice UP', 'Codice Censimp Impianto',
                'mean_pr_pvgis', 'kwp_used', 'actual_sum_kwh', 'expected_sum_kwh_pr1',
                'actual_over_expected_sum', 'energy_p99', 'energy_p99_over_kwp',
                'energy_max', 'energy_max_over_kwp', 'poa_kwm2_p99', 'poa_kwm2_max',
                'suggested_energy_multiplier', 'mean_pr_after_suggested_multiplier',
                'diagnostic_flags'
            ) if c in excluded_diag.columns
        ]
        print(excluded_diag[diag_cols].head(40).to_string(index=False))

_compact_report(OUT, top_n=20)